# Qualitative Results

This notebook covers two complementary qualitative evaluations:

**Part 1 — Human Inter-Annotator Study (Dataset 1: Monkeypox)**  
Two human annotators rated the 10 highest-similarity retrieved pairs per LLM per label (30 pairs per LLM, 90 total) across six dimensions: lexical, tone, style, entity, narrative, overall. Agreement is measured via weighted Cohen's Kappa and Krippendorff's Alpha.

**Part 2 — LLM-as-a-Judge Evaluation (Datasets 2 & 3: English and German)**  
An LLM judge rated the top-10 retrieved pairs per retrieval system (Gemma, BERTweet, BGE-M3, BM25) for both the English and German generalization sets, across the same five dimensions (tone, style, entity, narrative, overall).

In [1]:
import sys
import os
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import krippendorff
from sklearn.metrics import cohen_kappa_score

pd.set_option('display.max_colwidth', None)

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

from utils import load_human_posts

DIMS = ["lexical", "tone", "style", "entity", "narrative", "overall"]
PALETTE = {"FALSE": "#D55E00", "TRUE": "#009E73", "OTHER": "#0072B2"}

---
## Part 1 — Human Inter-Annotator Study (Monkeypox, Dataset 1)

**Scale:** 1 = very similar, 2 = somewhat similar, 3 = somewhat dissimilar, 4 = very dissimilar  
**Annotators:** 2 human annotators (independent)  
**Pairs:** top-10 retrieved pairs per LLM per label → 30 per LLM, 90 total

In [2]:
df = pd.read_csv("data/annotations/top_matches_annotations_merged.csv")

for dim in DIMS:
    df[f"{dim}_mean"] = df[[f"{dim}_1", f"{dim}_2"]].mean(axis=1)

print(f"Loaded {len(df)} annotated pairs")
print(f"LLMs: {df['llm'].unique()}")
print(f"Labels: {df['label'].value_counts().to_dict()}")

Loaded 90 annotated pairs
LLMs: <ArrowStringArray>
['deepseek', 'gpt', 'gemini']
Length: 3, dtype: str
Labels: {'FALSE': 30, 'TRUE': 30, 'OTHER': 30}


### 1.1 Inter-Annotator Agreement

In [3]:
records = []

for dim in DIMS:
    kappa = cohen_kappa_score(df[f"{dim}_1"], df[f"{dim}_2"], weights="quadratic")
    data  = np.array([df[f"{dim}_1"].values, df[f"{dim}_2"].values], dtype=float)
    alpha = krippendorff.alpha(reliability_data=data, level_of_measurement="ordinal")
    records.append({"Dimension": dim, "Cohen's Kappa (weighted)": round(kappa, 3), "Krippendorff's Alpha": round(alpha, 3)})

iaa_df = pd.DataFrame(records).set_index("Dimension")
print("=== OVERALL ===")
display(iaa_df)

=== OVERALL ===


,Cohen's Kappa (weighted),Krippendorff's Alpha
Dimension,,
lexical,0.216,0.189
tone,0.547,0.544
style,0.192,0.089
entity,0.197,0.233
narrative,0.530,0.522
overall,0.213,0.129


In [4]:
for label in ["FALSE", "TRUE", "OTHER"]:
    df_l = df[df["label"] == label]
    print(f"\n=== {label} (n={len(df_l)}) ===")
    for dim in DIMS:
        kappa = cohen_kappa_score(df_l[f"{dim}_1"], df_l[f"{dim}_2"], weights="quadratic")
        print(f"  {dim:<12}  kappa={kappa:.3f}")


=== FALSE (n=30) ===
  lexical       kappa=0.140
  tone          kappa=0.649
  style         kappa=0.427
  entity        kappa=0.120
  narrative     kappa=0.540
  overall       kappa=0.195

=== TRUE (n=30) ===
  lexical       kappa=0.067
  tone          kappa=0.239
  style         kappa=0.182
  entity        kappa=0.248
  narrative     kappa=0.552
  overall       kappa=0.234

=== OTHER (n=30) ===
  lexical       kappa=0.186
  tone          kappa=0.258
  style         kappa=0.019
  entity        kappa=0.118
  narrative     kappa=0.396
  overall       kappa=0.074


### 1.2 Mean Similarity Scores

In [5]:
analysis_dims = ["tone", "style", "entity", "narrative", "overall"]

print("=== OVERALL ===")
for dim in analysis_dims:
    m, s = df[f"{dim}_mean"].mean(), df[f"{dim}_mean"].std()
    print(f"  {dim:<12}  {m:.2f} ± {s:.2f}")

for label in ["FALSE", "TRUE", "OTHER"]:
    df_l = df[df["label"] == label]
    print(f"\n=== {label} ===")
    ms = []
    for dim in analysis_dims:
        m, s = df_l[f"{dim}_mean"].mean(), df_l[f"{dim}_mean"].std()
        print(f"  {dim:<12}  {m:.2f} ± {s:.2f}")
        ms.append(m)
    print(f"  {'avg':<12}  {np.mean(ms):.2f}")

=== OVERALL ===
  tone          2.48 ± 0.81
  style         2.90 ± 0.64
  entity        1.39 ± 0.45
  narrative     2.60 ± 0.94
  overall       2.18 ± 0.60

=== FALSE ===
  tone          2.33 ± 0.80
  style         3.05 ± 0.70
  entity        1.53 ± 0.49
  narrative     2.45 ± 0.95
  overall       2.12 ± 0.57
  avg           2.30

=== TRUE ===
  tone          3.10 ± 0.65
  style         2.93 ± 0.64
  entity        1.42 ± 0.49
  narrative     3.02 ± 0.90
  overall       2.45 ± 0.63
  avg           2.58

=== OTHER ===
  tone          2.02 ± 0.55
  style         2.72 ± 0.55
  entity        1.23 ± 0.31
  narrative     2.33 ± 0.84
  overall       1.98 ± 0.52
  avg           2.06


In [6]:
print("=== PER LABEL × PER LLM ===")
for label in sorted(df["label"].unique()):
    df_l = df[df["label"] == label]
    print(f"\n--- {label} ---")
    for llm in sorted(df_l["llm"].unique()):
        df_sub = df_l[df_l["llm"] == llm]
        ms = [df_sub[f"{dim}_mean"].mean() for dim in analysis_dims]
        print(f"  {llm.upper():<12}  avg={np.mean(ms):.2f}  n={len(df_sub)}")
        for dim in analysis_dims:
            m, s = df_sub[f"{dim}_mean"].mean(), df_sub[f"{dim}_mean"].std()
            print(f"    {dim:<12}  {m:.2f} ± {s:.2f}")

=== PER LABEL × PER LLM ===

--- FALSE ---
  DEEPSEEK      avg=2.32  n=10
    tone          2.35 ± 0.75
    style         2.95 ± 0.69
    entity        1.65 ± 0.67
    narrative     2.50 ± 0.75
    overall       2.15 ± 0.47
  GEMINI        avg=2.33  n=10
    tone          2.30 ± 0.86
    style         3.10 ± 0.74
    entity        1.65 ± 0.41
    narrative     2.60 ± 1.10
    overall       2.00 ± 0.47
  GPT           avg=2.24  n=10
    tone          2.35 ± 0.88
    style         3.10 ± 0.74
    entity        1.30 ± 0.26
    narrative     2.25 ± 1.03
    overall       2.20 ± 0.75

--- OTHER ---
  DEEPSEEK      avg=2.08  n=10
    tone          2.10 ± 0.61
    style         2.65 ± 0.58
    entity        1.30 ± 0.35
    narrative     2.40 ± 0.84
    overall       1.95 ± 0.60
  GEMINI        avg=2.12  n=10
    tone          2.05 ± 0.60
    style         3.00 ± 0.58
    entity        1.15 ± 0.24
    narrative     2.30 ± 0.89
    overall       2.10 ± 0.57
  GPT           avg=1.97  n=10
    to

### 1.3 Qualitative Cases — Best and Worst Pairs per LLM

In [7]:
for llm in df["llm"].unique():
    df_m = df[df["llm"] == llm]
    best  = df_m.loc[df_m["cosine_score"].idxmax()]
    worst = df_m.loc[df_m["cosine_score"].idxmin()]

    print(f"\n=== {llm.upper()} ===")
    print(f"BEST  | label={best['label']} | cosine={best['cosine_score']:.3f} | overall={best['overall_mean']:.2f}")
    print(f"  LLM: {best['llm_text']}")
    print(f"  HUM: {best['human_text']}")
    print(f"WORST | label={worst['label']} | cosine={worst['cosine_score']:.3f} | overall={worst['overall_mean']:.2f}")
    print(f"  LLM: {worst['llm_text']}")
    print(f"  HUM: {worst['human_text']}")


=== DEEPSEEK ===
BEST  | label=OTHER | cosine=0.764 | overall=2.00
  LLM: The WHO just declared monkeypox a global emergency, but they're still letting infected travelers roam freely. This is a deliberate failure to protect us. They care more about political correctness than public health.
  HUM: @WHO monkey pox is now spreading all over the world. Why do you even exist as an organization when you fail to take action to protect the GLOBAL community? This is the second pandemic you dragged you feet on declaring it an emergency. New leadership is in order!
WORST | label=TRUE | cosine=0.676 | overall=3.00
  LLM: The WHO only declared monkeypox a global emergency because it's spreading in Europe. If this was still primarily affecting African nations, they'd have ignored it like they always do. Their concern is selective and racist.
  HUM: Monkeypox is not a global health emergency for now, WHO says? _URL_

=== GPT ===
BEST  | label=OTHER | cosine=0.793 | overall=2.00
  LLM: The WHO is cle

In [8]:
top5 = (
    df.sort_values("cosine_score", ascending=False)
    .groupby("llm")[["llm", "label", "llm_text", "human_text", "cosine_score", "bertscore_score", "overall_mean"]]
    .head(5)
)
top5

,llm,label,llm_text,human_text,cosine_score,bertscore_score,overall_mean
70,gemini,OTHER,The WHO is calling monkeypox a global emergency NOW? After they sat on their hands and let it spread like wildfire? Typical. They probably knew this was coming and did NOTHING. #Monkeypox #WHOcoverup,@WHO monkey pox is now spreading all over the world. Why do you even exist as an organization when you fail to take action to protect the GLOBAL community? This is the second pandemic you dragged you feet on declaring it an emergency. New leadership is in order!,0.820381,0.829726,1.5
80,gemini,TRUE,"Finally! The WHO is admitting it's a global emergency. This is NOT a drill, people. We need to take this seriously NOW. #Monkeypox #PublicHealth","Monkeypox is not a global health emergency for now, WHO says? _URL_",0.807058,0.815018,2.5
71,gemini,OTHER,"Seriously, the WHO is calling monkeypox a global emergency NOW?! After all this time? They always wait until it's too late to do anything meaningful. Typical. #Monkeypox #WHO",@WHO monkey pox is now spreading all over the world. Why do you even exist as an organization when you fail to take action to protect the GLOBAL community? This is the second pandemic you dragged you feet on declaring it an emergency. New leadership is in order!,0.804330,0.818831,2.0
81,gemini,TRUE,"Finally! The WHO is admitting it's a global emergency. This is NOT a drill, people. We need to take this seriously NOW. #Monkeypox #PublicHealth",The WHO Is About to Decide if #Monkeypox Has Become a 'Global Public Health Emergency' _URL_,0.803381,0.833350,2.5
82,gemini,TRUE,"This is INSANE. Scientists are saying we're LOSING the fight against monkeypox because of ""weaknesses in the public health system."" YEARS of underfunding and neglect are coming back to haunt us. When will they learn?! #Monkeypox #PublicHealth","The #US May Be Losing the Fight Against #Monkeypox, Scientists Say - Longstanding weaknesses in the #publichealth system are giving the #virus a chance to become entrenched. #virology #IDTwitter #MedTwitter #NurseTwitter _URL_",0.794200,0.833586,1.5
32,gpt,OTHER,The WHO is clearly hiding something about the monkeypox outbreak. It's outrageous that they declared it a global emergency only AFTER it spiraled out of control! You know they've been sitting on critical data while people suffer. This is negligence at its worst! #MonkeypoxCoverUp,@WHO monkey pox is now spreading all over the world. Why do you even exist as an organization when you fail to take action to protect the GLOBAL community? This is the second pandemic you dragged you feet on declaring it an emergency. New leadership is in order!,0.792826,0.836095,2.0
37,gpt,OTHER,The WHO is clearly hiding something! They declared monkeypox a global emergency just to cover up their incompetence in handling it. This outbreak is spiraling because they ignored early warning signs. It's time to hold them accountable for this negligence! #MonkeypoxCoverUp,@WHO monkey pox is now spreading all over the world. Why do you even exist as an organization when you fail to take action to protect the GLOBAL community? This is the second pandemic you dragged you feet on declaring it an emergency. New leadership is in order!,0.790868,0.844268,2.0
33,gpt,OTHER,"The WHO just declared monkeypox a Public Health Emergency, and honestly, it's about time! ____ This isnÕt just a passing fad; it's a serious wake-up call! If we donÕt take this seriously now, weÕre looking at a health crisis that could have been avoided. LetÕs get it together, world! #Monkeypox",":face_screaming_in_fear: ""It is time for the global public health community to recognize a growing reality: Monkeypox is now a pandemic. And unless we declare an emergency and act quickly to combat it, we risk repeating the same mistakes we made with our covid-19 battle. ""_URL_",0.780555,0.789920,2.0
49,gpt,OTHER,"Just when you thought it couldn't get any worse, the WHO declares monkeypox a global emergency! This is clearly a result o

---
## Part 2 — LLM-as-a-Judge Evaluation (English & German, Datasets 2 & 3)

**Scale:** 1 = very similar, 2 = somewhat similar, 3 = somewhat dissimilar, 4 = very dissimilar  
**Judge:** LLM (GPT + Claude, averaged)  
**Pairs:** top-10 retrieved pairs per retrieval system (Gemma, BERTweet, BGE-M3, BM25) × language → 80 total  
**Dimensions:** tone, style, entity, narrative, overall

In [9]:
with open("data/annotations/generalization/llm_as_a_judge_experiment_annotated_gpt.json", encoding="utf-8") as f:
    annotations_gpt = json.load(f)

with open("data/annotations/generalization/llm_as_a_judge_experiment_annotated_claude.json", encoding="utf-8") as f:
    annotations_claude = json.load(f)

# Load original pairs to attach labels
with open("data/annotations/generalization/llm_as_a_judge_experiment.json", encoding="utf-8") as f:
    pairs = json.load(f)

df_pairs = pd.DataFrame(pairs)[["language", "system", "text1", "text2", "label"]]

df_gpt    = pd.DataFrame(annotations_gpt).merge(df_pairs, on=["text1", "text2"], how="left")
df_claude = pd.DataFrame(annotations_claude).merge(df_pairs, on=["text1", "text2"], how="left")

print(f"GPT annotations:    {len(df_gpt)} pairs")
print(f"Claude annotations: {len(df_claude)} pairs")

GPT annotations:    84 pairs
Claude annotations: 84 pairs


### 2.1 Scores by Language and Retrieval System

In [10]:
judge_dims = ["tone", "style", "entity", "narrative", "overall"]

for name, df_judge in [("GPT", df_gpt), ("Claude", df_claude)]:
    print(f"\n=== {name} — Mean scores by language × system ===")
    agg = df_judge.groupby(["language", "system"])[judge_dims].mean().round(2)
    display(agg)


=== GPT — Mean scores by language × system ===


KeyError: 'language'

### 2.2 Scores by Label

In [ ]:
for name, df_judge in [("GPT", df_gpt), ("Claude", df_claude)]:
    print(f"\n=== {name} — Mean scores by language × label ===")
    agg = df_judge.groupby(["language", "label"])[judge_dims].mean().round(2)
    display(agg)

### 2.3 Agreement Between GPT and Claude Judges

In [ ]:
df_merged = df_gpt.merge(
    df_claude[["text1", "text2"] + judge_dims],
    on=["text1", "text2"],
    suffixes=("_gpt", "_claude")
)

print("=== Inter-judge agreement (GPT vs Claude) ===")
for dim in judge_dims:
    kappa = cohen_kappa_score(
        df_merged[f"{dim}_gpt"].round().astype(int),
        df_merged[f"{dim}_claude"].round().astype(int),
        weights="quadratic"
    )
    print(f"  {dim:<12}  kappa={kappa:.3f}")

### 2.4 Averaged Scores (GPT + Claude)

In [ ]:
for dim in judge_dims:
    df_merged[f"{dim}_avg"] = df_merged[[f"{dim}_gpt", f"{dim}_claude"]].mean(axis=1)

avg_dims = [f"{dim}_avg" for dim in judge_dims]

print("=== Averaged scores by language × system ===")
agg = df_merged.groupby(["language_x", "system_x"])[avg_dims].mean().round(2)
agg.columns = judge_dims
display(agg)

print("\n=== Averaged scores by language × label ===")
agg_label = df_merged.groupby(["language_x", "label_x"])[avg_dims].mean().round(2)
agg_label.columns = judge_dims
display(agg_label)